# Interactive controls: a slider that re-runs the analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GMOD/jbrowse-anywidget/blob/main/examples/09_interactive_controls.ipynb)

The view is wired to a live kernel, so a widget control can **re-run the computation** and repaint the track — not just filter a static file. Here an `ipywidgets` slider sets the significance threshold for a differential-expression call; moving it reclassifies every gene in Python and pushes the updated track. The genome view and the control sit side by side, both driven from the same notebook state.

In [ ]:
# Install only if not already available (e.g. in Colab). The GitHub install
# needs no JS toolchain — the built widget bundle is committed in the repo. A
# local editable install is used as-is. (Swap to `jbrowse-anywidget` once it's
# published to PyPI.)
try:
    import jbrowse_anywidget  # noqa: F401
except ImportError:
    %pip install -q "jbrowse-anywidget @ git+https://github.com/GMOD/jbrowse-anywidget" pandas numpy scipy

# Colab requires this to render third-party (anywidget) widgets:
try:
    from google.colab import output

    output.enable_custom_widget_manager()
except ImportError:
    pass

## The analysis

The same small DE table as the DE example — genes with a log2 fold-change and a p-value. `classify` is the part a slider re-runs: it labels each gene up / down / not-significant at a chosen p-value cutoff. Swap in your own DESeq2/edgeR table joined to coordinates.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind

rng = np.random.default_rng(7)
n_genes, n_rep = 80, 4
chrom, gene_len, gap = "7", 6_000, 40_000
starts = 1_000_000 + np.arange(n_genes) * gap

base = rng.uniform(20, 400, n_genes)
true_lfc = np.zeros(n_genes)
up = rng.choice(n_genes, 6, replace=False)
down = rng.choice(np.setdiff1d(np.arange(n_genes), up), 6, replace=False)
true_lfc[up] = rng.uniform(1.5, 3.0, up.size)
true_lfc[down] = -rng.uniform(1.5, 3.0, down.size)

ctrl = rng.poisson(base[:, None], size=(n_genes, n_rep))
treat = rng.poisson((base * 2.0**true_lfc)[:, None], size=(n_genes, n_rep))
lc, lt = np.log2(ctrl + 1), np.log2(treat + 1)
lfc = lt.mean(1) - lc.mean(1)
pval = ttest_ind(lt, lc, axis=1, equal_var=False).pvalue

de = pd.DataFrame(
    {
        "chrom": chrom,
        "start": starts,
        "end": starts + gene_len,
        "name": [f"GENE{i:04d}" for i in range(n_genes)],
        "log2fc": lfc.round(2),
        "pvalue": pval,
    }
)


def classify(pvalue_cutoff, lfc_cutoff=1.0):
    sig = np.where(
        (de.pvalue < pvalue_cutoff) & (de.log2fc.abs() > lfc_cutoff),
        np.where(de.log2fc > 0, "up", "down"),
        "ns",
    )
    return de.assign(sig=sig)


classify(0.01).sig.value_counts()

## Wire a slider to the view

`render` reruns `classify` at the slider's cutoff and replaces the track (clearing first, so moving the slider repaints in place rather than stacking tracks). `slider.observe` calls it on every change — including a programmatic one, which is how this runs headless below. Drag the slider and the genes recolor live.

In [ ]:
import ipywidgets as widgets

from jbrowse_anywidget import LinearGenomeView, make_assembly

grch38 = make_assembly(
    "GRCh38",
    "https://jbrowse.org/genomes/GRCh38/fasta/GRCh38.fa.gz",
    aliases=["hg38"],
)
view = LinearGenomeView(assembly=grch38, location="7:1,000,000..4,300,000")

COLOR = "jexl:get(feature,'sig') == 'up' ? '#c62828' : get(feature,'sig') == 'down' ? '#1565c0' : '#cfcfcf'"


def render(pvalue_cutoff):
    view.tracks = []  # replace, don't stack
    view.add_features(
        classify(pvalue_cutoff),
        name=f"DE (p < {pvalue_cutoff:g})",
        track_id="de",
        color=COLOR,
    )


slider = widgets.FloatLogSlider(
    value=0.01, base=10, min=-4, max=-1, step=0.2, description="p <",
)
slider.observe(lambda change: render(change["new"]), "value")
render(slider.value)

widgets.VBox([slider, view])

Setting the slider from code fires the same observer, so this tightens the threshold and repaints the track without any manual interaction:

In [ ]:
slider.value = 1e-4
print("significant now:", int((classify(slider.value).sig != "ns").sum()), "genes")